## Lab 4 v2: Deploy Complete Product Launch System to Production - Multi-agent approach

### Overview

Deploy the complete multi-agent product launch system with all components:
- **Multi-Agent System** (Lab 1) - 5 specialized agents + orchestrator
- **Memory** (Lab 2) - Persistent conversation history
- **Gateway Tools** (Lab 3) - Centralized tool management
- **Code Interpreter** (Lab 3ii) - Financial calculations
- **Browser** (Lab 3i) - Competitive research

### Architecture

```
[AgentCore Runtime]
    ↓
[Multi-Agent Orchestrator]
    ↓
[Market | Compliance | Strategy | Marketing | Deployment]
    ↓
[Gateway Tools] + [Memory] + [Code Interpreter] + [Browser]
```

### Prerequisites
- Completed Labs 1, 2, 3
- Docker installed
- AWS account with permissions

### Step 1: Import and Verify Components

In [ ]:
import os
import sys
import boto3
from boto3.session import Session

# Add current directory to path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

boto_session = Session()
region = boto_session.region_name

print(f"✅ Region: {region}")

### Step 2: Verify Memory (Lab 2)

In [ ]:
from lab_helpers.utils import get_ssm_parameter

try:
    memory_id = get_ssm_parameter("/app/productlaunch/agentcore/memory_id")
    print(f"✅ Memory found: {memory_id}")
except:
    print("⚠️ Memory not found - run Lab 2 first")

### Step 3: Verify Gateway (Lab 3)

In [ ]:
try:
    gateway_id = get_ssm_parameter("/app/productlaunch/agentcore/gateway_id")
    print(f"✅ Gateway found: {gateway_id}")
except:
    print("⚠️ Gateway not found - run Lab 3 first")

### Step 4: Create Runtime Entrypoint

This file will be deployed to AgentCore Runtime.

In [ ]:
%%writefile lab_helpers/lab4_runtime_v2.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from lab_helpers.ai_multi_agent_system import get_ai_orchestrator_agent

# Get the multi-agent orchestrator
orchestrator = get_ai_orchestrator_agent()

# Initialize AgentCore Runtime App
app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload):
    """AgentCore Runtime entrypoint - routes to multi-agent orchestrator"""
    user_input = payload.get("prompt", "")
    response = orchestrator(user_input)
    return response

if __name__ == "__main__":
    app.run()

### Step 5: Setup Cognito Authentication

In [ ]:
from lab_helpers.utils import get_or_create_cognito_pool, reauthenticate_user

print("Setting up Cognito user pool...")
cognito_config = get_or_create_cognito_pool()
print(f"✅ Cognito configured: {cognito_config.get('user_pool_id')}")

### Step 6: Create IAM Execution Role

In [ ]:
from lab_helpers.utils import create_agentcore_runtime_execution_role

execution_role_arn = create_agentcore_runtime_execution_role()
print(f"✅ Execution role: {execution_role_arn}")

### Step 7: Configure Runtime

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="lab_helpers/lab4_runtime_v2.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="productlaunchagent",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
)

print("✅ Runtime configured")

### Step 8: Verify Dockerfile

In [ ]:
import os
dockerfile_path = './Dockerfile'

with open(dockerfile_path, 'r') as f:
    content = f.read()
    
if 'lab_helpers' in content:
    print('✅ Dockerfile includes lab_helpers directory')
else:
    print('❌ Dockerfile missing lab_helpers')

### Step 9: Clear Old Agent ID (if re-launching)

In [ ]:
import yaml

config_path = agentcore_runtime._config_path
print(f"Config: {config_path}")

if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    if 'bedrock_agentcore' in config and 'agent_id' in config['bedrock_agentcore']:
        old_id = config['bedrock_agentcore']['agent_id']
        del config['bedrock_agentcore']['agent_id']
        
        with open(config_path, 'w') as f:
            yaml.dump(config, f)
        print(f"✅ Removed old agent ID: {old_id}")
    else:
        print("✅ No old agent ID found")

### Step 10: Launch to Production

In [ ]:
# Open a Terminal and run the command - rm .bedrock_agentcore.yaml    if the below cell fails

In [ ]:
from lab_helpers.utils import put_ssm_parameter

launch_result = agentcore_runtime.launch()
print(f"✅ Launch completed: {launch_result.agent_arn}")

put_ssm_parameter("/app/productlaunch/agentcore/runtime_arn", launch_result.agent_arn)

### Step 11: Test the Production Agent

In [ ]:
import uuid
from IPython.display import Markdown, display

session_id = uuid.uuid4()

bearer_token = reauthenticate_user(
    cognito_config.get("client_id"),
    cognito_config.get("client_secret")
)

# Test full product launch
response = agentcore_runtime.invoke(
    {"prompt": "Launch a new auto loan product for millennials at 5.99% APR. Include market research, strategy, and marketing."},
    bearer_token=bearer_token,
    session_id=str(session_id)
)

response

### Step 12: Test Individual Capabilities

In [ ]:
# Test market research
session_id2 = uuid.uuid4()
response = agentcore_runtime.invoke(
    {"prompt": "Do market research for personal loans in the US"},
    bearer_token=bearer_token,
    session_id=str(session_id2)
)
display(Markdown(response.get('response', response)))

In [ ]:
# Test marketing
session_id3 = uuid.uuid4()
response = agentcore_runtime.invoke(
    {"prompt": "Create a marketing poster for a 0% APR credit card"},
    bearer_token=bearer_token,
    session_id=str(session_id3)
)
display(Markdown(response.get('response', response)))

## 🎉 Lab 4 v2 Complete!

You've deployed a complete production system with:
- ✅ Multi-agent orchestration (5 specialized agents)
- ✅ Persistent memory across sessions
- ✅ Centralized gateway tools
- ✅ Code interpreter for calculations
- ✅ Browser automation for research
- ✅ Auto-scaling runtime
- ✅ Full observability

**Next:** [Lab 5: Build User Interface →](lab-05-add-frontend.ipynb)